In [5]:


from getpass import getpass
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent



GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")



llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


sample_text = """
LangChain is a framework for developing applications powered by
large language models.

It provides tools for building AI applications using language models,
prompts, chains, agents, and retrieval augmented generation.

LangChain was created by Harrison Chase.

LangChain helps developers connect language models with external
data sources, APIs, databases, and other tools.

Retrieval Augmented Generation (RAG) is a technique that retrieves
relevant information from external documents and uses that information
to generate accurate answers.

FAISS is a vector database library used for efficient similarity search.

Embeddings convert text into numerical vector representations.
These vectors help identify semantically similar documents.
"""

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("Sample text file created successfully!")



with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()


splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(text_data)

print("Total chunks:", len(texts))



embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)



vectorstore = FAISS.from_texts(
    texts=texts,
    embedding=embedding
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("FAISS vector database created successfully!")



@tool
def LangChainRetriever(query: str) -> str:
    """
    Use this tool to answer questions about LangChain,
    its features, its creator, embeddings, FAISS, and RAG.
    """


    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information found in the knowledge base."

    # Combine retrieved documents
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer using Gemini and retrieved context
    prompt = f"""
    Answer the question using only the context provided below.

    If the answer is not available in the context,
    clearly say that the information is not available.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    answer = llm.invoke(prompt)

    return answer.content


checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "rag-demo-1"
    }
}



agent = create_agent(
    model=llm,
    tools=[LangChainRetriever],
    system_prompt="""
    You are a helpful LangChain RAG assistant.

    Use the LangChainRetriever tool whenever the user asks
    questions about information in the knowledge base.

    Provide clear and concise answers based on retrieved context.
    Do not invent information.
    """,
    checkpointer=checkpointer
)

print("RAG Agent created successfully!")



response1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    config=thread_config
)

print("\nQuestion 1: What is LangChain?")
print("Answer:", response1["messages"][-1].content)


response2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    config=thread_config
)

print("\nQuestion 2: Who created it?")
print("Answer:", response2["messages"][-1].content)



response3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain its use in AI workflows."
            }
        ]
    },
    config=thread_config
)

print("\nQuestion 3: Explain its use in AI workflows.")
print("Answer:", response3["messages"][-1].content)


print("\n========== CONVERSATION HISTORY ==========\n")

for message in response3["messages"]:
    if message.type in ["human", "ai"]:
        print(f"{message.type.upper()}: {message.content}\n")


Enter Gemini API Key: ··········
Sample text file created successfully!
Total chunks: 3
FAISS vector database created successfully!
RAG Agent created successfully!


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Question 1: What is LangChain?
Answer: [{'type': 'text', 'text': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'extras': {'signature': 'EokBCoYBAWkUfRPbNcGxkluoDfpg3nfnAiD3K3kfXCuoSgamYjk5bC5yQu5sl7aFB/bl30nIMq4AGJWBiU8GyEtAEUctoBKSGESVt1IwzBPUpJB0jAYU5t0xeoawmjbiCptIkA7DGff6VNi1NPPGoIHNHhpAyOH9Od6CrMTkSYZNs4f8avWlOJwGXmU='}}]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Question 2: Who created it?
Answer: [{'type': 'text', 'text': 'LangChain was created by Harrison Chase.', 'extras': {'signature': 'EokBCoYBAWkUfRMc1TxaA+AIXJN0lqcrdsZ9AG4QhFPSJ+CUs/ZnYmZirWISl2yG7Yu1YW5HcRdIeu1Ol2q71dh5gj4MMHCM12Et+AYvH9FAZPGTslp7Cd0s2BqoL+a2T9n7qqCX5R7enHnGZ3Ahlj1gdRjoanc0xsFJs9UI8WyQQCIS/jmyG/eyOr8='}}]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_


Question 3: Explain its use in AI workflows.
Answer: [{'type': 'text', 'text': 'Based on the available information, LangChain is a framework for developing applications powered by large language models, but specific details on its use in AI workflows, RAG, embeddings, or FAISS are not included in the knowledge base.', 'extras': {'signature': 'EokBCoYBAWkUfRPxE7KKZTyp41+A7MnhPZ/tOD66MsvfFD3We5GgPMYsJfz0FK6hZmNhK/NiWvoif0FhgHBbGZe1KjKcFr8XMpQiaXV6fGBxJ9khOd8m/hUcytKWBOskTUY4Os2cxXLbcsld0RcpSvuT7ARPfwfKqQA1aZ2fHYZWabs95rgc5pazzdQ='}}]

========== CONVERSATION HISTORY ==========

HUMAN: What is LangChain?

AI: []

AI: [{'type': 'text', 'text': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'extras': {'signature': 'EokBCoYBAWkUfRPbNcGxkluoDfpg3nfnAiD3K3kfXCuoSgamYjk5bC5yQu5sl7aFB/bl30nIMq4AGJWBiU8GyEtAEUctoBKSGESVt1IwzBPUpJB0jAYU5t0xeoawmjbiCptIkA7DGff6VNi1NPPGoIHNHhpAyOH9Od6CrMTkSYZNs4f8avWlOJwGXmU='}}]

HUMAN: Who created it?

AI: []

AI:

In [4]:
%pip install -q --force-reinstall --no-deps \
    "google-auth==2.49.0" \
    "google-genai==2.34.0"

In [4]:
%pip install -q \
    "langchain==1.3.11" \
    "langchain-google-genai==4.4.3" \
    "langchain-community==0.4.2" \
    "langchain-text-splitters" \
    "langgraph" \
    "faiss-cpu" \
    "filetype"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
